# Semantics: open vs closed world, NAF, WFS, and ASP

One innocent-looking symbol — *negation* — hides some of the deepest design choices in logic
programming. When you write "**not** `Flies(tweety)`", what does it mean?

- Is it **false**, or merely **not known to be true**?
- If you learn a new fact tomorrow, can a conclusion you drew today be **withdrawn**?
- If two conclusions are locked in a mutual "each is true unless the other is", is the answer
  **two possible worlds**, **no world at all**, or **"undefined"**?

Different logics answer these questions differently. This notebook is an *executable* tour of
those answers. Every claim below is backed by a solver you can re-run — no hand-waving.

We cover four ideas and the relationships between them:

| Idea | One-line intuition | Illustrated with |
|------|--------------------|------------------|
| **Open-World Assumption (OWA)** | absence of a fact means *unknown* | Z3 (classical FOL) |
| **Closed-World Assumption (CWA)** | absence of a fact means *false* | Clingo, Datalog |
| **Negation as Failure (NAF)** | `not p` succeeds if `p` can't be proven | Clingo |
| **Well-Founded Semantics (WFS)** | three-valued: true / false / **undefined** | reference implementation |
| **Answer Set Programming (ASP)** | stable models: enumerate self-consistent worlds | Clingo |

These correspond directly to profile classes shipped in `typedlogic.profiles`
(`OpenWorld`, `ClosedWorld`, `WellFoundedSemantics`, `ClassicPrologNegationAsFailure`,
`AnswerSetProgramming`), which we revisit at the end.

## Setup

This notebook uses the Z3 and Clingo integrations:

```bash
pip install 'typedlogic[z3,clingo]'
```

Z3 is a classical first-order theorem prover — it embodies the **open-world**, **monotonic**
view. Clingo is an **Answer Set Programming** solver — it embodies the **closed-world**,
**non-monotonic** view. Putting them side by side is the whole point.

To keep everything honest and self-contained, we define each logical theory as a short block of
`typedlogic` Python source and parse it with `PythonParser` in the notebook itself — nothing is
hidden in helper files.

In [1]:
from typedlogic import Term, Not
from typedlogic.parsers.pyparser import PythonParser
from typedlogic.integrations.solvers.z3 import Z3Solver
from typedlogic.integrations.solvers.clingo import ClingoSolver

parser = PythonParser()

def theory(src: str):
    "Parse a typedlogic source string into a Theory."
    return parser.parse(src)

## 1. Open world vs closed world

Consider a tiny flight database. There is a direct flight SFO→JFK and JFK→LHR. There is **no**
row for SFO→LHR.

In [2]:
flights_src = """
from dataclasses import dataclass
from typedlogic import FactMixin

@dataclass(frozen=True)
class DirectFlight(FactMixin):
    src: str
    dst: str
"""

flights = theory(flights_src)
facts = [Term("DirectFlight", "sfo", "jfk"), Term("DirectFlight", "jfk", "lhr")]

**The question:** *Is there a direct flight from SFO to LHR?* We never asserted one. What should
a reasoner say?

### Closed world (Clingo)

A closed-world reasoner treats the database as complete. Anything not derivable is taken to be
false. Clingo returns exactly one model — the facts we gave it — and `DirectFlight(sfo, lhr)`
is simply absent, i.e. **false**.

In [3]:
s = ClingoSolver()
s.add(flights)
for f in facts:
    s.add(f)

model = list(s.models())[0]
present = {t.values for t in model.iter_retrieve("DirectFlight")}
print("Direct flights in the (single) closed-world model:")
for t in sorted(present):
    print("  ", t)
print()
print("Is DirectFlight(sfo, lhr) in the model?", ("sfo", "lhr") in present)
print("=> Under CWA, 'no row' means FALSE.")

Direct flights in the (single) closed-world model:
   ('jfk', 'lhr')
   ('sfo', 'jfk')

Is DirectFlight(sfo, lhr) in the model? False
=> Under CWA, 'no row' means FALSE.


### Open world (Z3)

A classical, open-world reasoner treats the database as *partial*. The absence of SFO→LHR tells
us nothing. We can prove this operationally: a fact is **entailed** only if its negation is
*inconsistent* with the knowledge base. So we check satisfiability **both ways**.

In [4]:
# KB + DirectFlight(sfo,lhr): is it consistent?
s = Z3Solver(); s.add(flights)
for f in facts: s.add(f)
s.add(Term("DirectFlight", "sfo", "lhr"))
with_flight = s.check().satisfiable

# KB + NOT DirectFlight(sfo,lhr): is it consistent?
s = Z3Solver(); s.add(flights)
for f in facts: s.add(f)
s.add(Not(Term("DirectFlight", "sfo", "lhr")))
without_flight = s.check().satisfiable

print("KB +  DirectFlight(sfo,lhr) is satisfiable:", with_flight)
print("KB + ~DirectFlight(sfo,lhr) is satisfiable:", without_flight)
print()
print("Both are satisfiable => neither the flight nor its absence is entailed.")
print("=> Under OWA, 'no row' means UNKNOWN.")

KB +  DirectFlight(sfo,lhr) is satisfiable: True
KB + ~DirectFlight(sfo,lhr) is satisfiable: True

Both are satisfiable => neither the flight nor its absence is entailed.
=> Under OWA, 'no row' means UNKNOWN.


This is the fundamental split:

- **Closed world**: what is not known to be true is **false**. Great for databases, configuration,
  policy checks — settings where you *do* have all the relevant facts.
- **Open world**: what is not known to be true is **unknown**. Right for the semantic web,
  ontologies, and incomplete knowledge — where new facts can always arrive (this is the OWL/RDF
  world; see the OWL-DL tutorial).

Neither is "correct" in the abstract; they answer different questions. The bugs happen when you
assume one and your tool implements the other.

## 2. Negation as failure, and non-monotonicity

The closed-world assumption gives us a *cheap* form of negation: `not p` succeeds precisely when
we **fail to prove** `p`. This is **negation as failure (NAF)** — the negation of Prolog, Datalog,
and ASP. In `typedlogic`'s Python syntax NAF is written with a unary minus (`-p`), distinct from
classical/strong negation (`~p`).

The classic use is **default reasoning**: *birds fly, unless they are abnormal.*

In [5]:
birds_src = """
from dataclasses import dataclass
from typedlogic import FactMixin
from typedlogic.decorators import axiom

@dataclass(frozen=True)
class Bird(FactMixin):
    name: str

@dataclass(frozen=True)
class Penguin(FactMixin):
    name: str

@dataclass(frozen=True)
class Abnormal(FactMixin):
    name: str

@dataclass(frozen=True)
class Flies(FactMixin):
    name: str

@axiom
def penguins_are_abnormal(x: str):
    if Penguin(x):
        assert Abnormal(x)

@axiom
def birds_fly_by_default(x: str):
    # NAF: a bird flies UNLESS it can be shown abnormal (note the unary minus)
    if Bird(x) and -Abnormal(x):
        assert Flies(x)
"""
birds = theory(birds_src)


def who_flies(extra_facts=()):
    s = ClingoSolver()
    s.add(birds)
    s.add(Term("Bird", "tweety"))
    s.add(Term("Bird", "opus"))
    s.add(Term("Penguin", "opus"))
    for f in extra_facts:
        s.add(f)
    model = list(s.models())[0]
    return sorted(t.values[0] for t in model.iter_retrieve("Flies"))


print("Who flies by default?", who_flies())

Who flies by default? ['tweety']


`tweety` flies; `opus` does not (it's a penguin, hence abnormal). Nobody told us tweety is
*normal* — the reasoner **assumed** it because it couldn't prove otherwise. That assumption is
the closed world at work.

Now the crucial property. Watch what happens when we *add* a fact.

In [6]:
print("Before: ", who_flies())
print("Add Penguin(tweety) ...")
print("After:  ", who_flies(extra_facts=[Term("Penguin", "tweety")]))
print()
print("Learning MORE made us conclude LESS. This is NON-MONOTONICITY.")

Before:  ['tweety']
Add Penguin(tweety) ...
After:   []

Learning MORE made us conclude LESS. This is NON-MONOTONICITY.


Adding knowledge **retracted** a conclusion. Classical logic can never do this: it is
**monotonic** — the set of entailed facts only ever grows. NAF buys expressiveness (defaults,
exceptions, common-sense reasoning) at the price of monotonicity.

### The same rule, read classically (Z3)

What if we read "birds fly unless abnormal" with *classical* negation (`~`) instead? Then the
rule is the material implication `Bird(x) ∧ ¬Abnormal(x) → Flies(x)`. Under the open world, we
*cannot* conclude `Flies(tweety)` from `Bird(tweety)` alone — because `¬Abnormal(tweety)` is not
entailed (it's merely unknown).

In [7]:
birds_classical_src = """
from dataclasses import dataclass
from typedlogic import FactMixin
from typedlogic.decorators import axiom

@dataclass(frozen=True)
class Bird(FactMixin):
    name: str

@dataclass(frozen=True)
class Abnormal(FactMixin):
    name: str

@dataclass(frozen=True)
class Flies(FactMixin):
    name: str

@axiom
def birds_fly_classically(x: str):
    # classical/strong negation (~), material implication
    if Bird(x) and ~Abnormal(x):
        assert Flies(x)
"""
birds_classical = theory(birds_classical_src)

# Is Flies(tweety) entailed from Bird(tweety) alone?
s = Z3Solver(); s.add(birds_classical)
s.add(Term("Bird", "tweety"))
s.add(Not(Term("Flies", "tweety")))
print("KB + ~Flies(tweety) satisfiable?", s.check().satisfiable,
      " => Flies(tweety) NOT entailed (abnormality is unknown, not false)")

# Now assert tweety is not abnormal, classically:
s = Z3Solver(); s.add(birds_classical)
s.add(Term("Bird", "tweety"))
s.add(Not(Term("Abnormal", "tweety")))
s.add(Not(Term("Flies", "tweety")))
print("KB + ~Abnormal(tweety) + ~Flies(tweety) satisfiable?", s.check().satisfiable,
      " => now Flies(tweety) IS entailed")

KB + ~Flies(tweety) satisfiable? True  => Flies(tweety) NOT entailed (abnormality is unknown, not false)
KB + ~Abnormal(tweety) + ~Flies(tweety) satisfiable? False  => now Flies(tweety) IS entailed


So the *very same English sentence* means different things depending on the negation you pick:

- **NAF** (`-Abnormal`): "assume normal unless proven abnormal" → concludes `Flies(tweety)`
  immediately, but retracts it later. Non-monotonic.
- **Classical** (`~Abnormal`): "flies only if *provably* not abnormal" → stays agnostic until you
  supply the missing premise. Monotonic.

Choosing the wrong one is a real and common modelling bug.

## 3. When NAF loops: stable models (ASP)

Default reasoning was *stratified* — negation never depended on itself. Once we allow negation to
loop back on itself, "what does `not p` mean" stops having an obvious answer, and the different
semantics genuinely diverge. This is where **WFS** and **ASP** part ways.

**Answer Set Programming** answers with **stable models** (answer sets): each is a self-consistent
world — a set of atoms that reproduces exactly itself when you apply the rules under its own
negative assumptions. There can be several, or none.

### Even loop → two worlds

```
p :- not q.
q :- not p.
```

"p holds unless q does, and q holds unless p does." Two coherent stories: *p and not q*, or
*q and not p*.

In [8]:
from typedlogic import Implies, NegationAsFailure
from typedlogic.datamodel import PredicateDefinition, Theory as _Theory

def prop_theory(names, rules):
    "Build a propositional theory: names are 0-ary predicates, rules are (head, [naf_atoms])."
    th = _Theory(predicate_definitions=[PredicateDefinition(n, {}) for n in names])
    for head, naf_atoms in rules:
        body = NegationAsFailure(Term(naf_atoms[0]))
        for a in naf_atoms[1:]:
            body = body & NegationAsFailure(Term(a))
        th.add(Implies(body, Term(head)))
    return th

even = prop_theory(["p", "q"], [("p", ["q"]), ("q", ["p"])])
s = ClingoSolver(); s.add(even)
models = list(s.models())
print("Number of stable models (answer sets):", len(models))
for i, m in enumerate(models):
    print(f"  model {i}:", sorted(str(t) for t in m.ground_terms))

Number of stable models (answer sets): 2
  model 0: ['q']
  model 1: ['p']


### Odd loop → no world

```
p :- not p.
```

"p holds if p does not." No set of atoms can reproduce itself: assume `p` false and the rule
forces it true; assume it true and the rule no longer fires. **Zero** stable models — the program
is *inconsistent* in ASP.

In [9]:
odd = prop_theory(["p"], [("p", ["p"])])
s = ClingoSolver(); s.add(odd)
print("Satisfiable?", s.check().satisfiable)
print("Number of stable models:", len(list(s.models())))
print("=> An odd negative loop has NO answer set.")

Satisfiable? False
Number of stable models: 0
=> An odd negative loop has NO answer set.


Stable-model semantics is decisive and credulous: it hands you every self-consistent world, or
refuses when none exists. That's ideal for combinatorial search (planning, configuration,
graph coloring), but the "all or nothing" reaction to a loop can be surprisingly brittle.

## 4. A gentler answer: well-founded semantics

**Well-founded semantics (WFS)** refuses to guess. Instead of enumerating worlds it computes a
*single* **three-valued** model: every atom is **true**, **false**, or **undefined**. Paradoxical
loops land in *undefined* rather than exploding into many models or none.

There is no bundled WFS solver in `typedlogic`, so — to stay honest — here is a compact, fully
transparent implementation of the standard **Van Gelder alternating fixpoint**. It operates on
propositional normal programs and is short enough to read end to end.

In [10]:
from typing import NamedTuple, FrozenSet, Set

class WRule(NamedTuple):
    head: str
    naf: FrozenSet[str]   # negation-as-failure body atoms (positive-body-free for these demos)

def _gamma(rules, I: Set[str]) -> Set[str]:
    """Least model of the Gelfond-Lifschitz reduct w.r.t. the true-atom set I.
    `not a` holds iff a not in I, so a rule survives iff its naf atoms all avoid I."""
    heads = [r.head for r in rules if r.naf.isdisjoint(I)]
    return set(heads)

def _fixpoint(rules, start: Set[str]) -> Set[str]:
    cur = set(start)
    while True:
        nxt = _gamma(rules, _gamma(rules, cur))   # gamma is antimonotone; gamma^2 is monotone
        if nxt == cur:
            return cur
        cur = nxt

def well_founded_model(rules):
    atoms: Set[str] = set()
    for r in rules:
        atoms.add(r.head); atoms |= r.naf
    true = _fixpoint(rules, set())         # least fixpoint of gamma^2  -> definitely true
    non_false = _fixpoint(rules, atoms)    # greatest fixpoint of gamma^2 -> possibly true
    return true, atoms - non_false, non_false - true   # (true, false, undefined)

def show_wfs(name, rules):
    t, f, u = well_founded_model(rules)
    print(f"{name}: TRUE={sorted(t)}  FALSE={sorted(f)}  UNDEFINED={sorted(u)}")

Now run WFS on the exact same loops that gave ASP two models / zero models:

In [11]:
even_wfs = [WRule("p", frozenset({"q"})), WRule("q", frozenset({"p"}))]
odd_wfs  = [WRule("p", frozenset({"p"}))]

show_wfs("even loop (p:-not q, q:-not p)", even_wfs)
show_wfs("odd loop  (p:-not p)          ", odd_wfs)

even loop (p:-not q, q:-not p): TRUE=[]  FALSE=[]  UNDEFINED=['p', 'q']
odd loop  (p:-not p)          : TRUE=[]  FALSE=[]  UNDEFINED=['p']


Compare the two semantics on the identical programs:

| Program | ASP (stable models) | WFS (well-founded model) |
|---------|---------------------|--------------------------|
| `p:-not q. q:-not p.` | **two** models: `{p}` and `{q}` | one model: `p`, `q` both **undefined** |
| `p:-not p.` | **no** model (inconsistent) | one model: `p` **undefined** |

ASP is *credulous and brave*: it commits to concrete worlds and enumerates them. WFS is *skeptical
and cautious*: where the program is genuinely ambiguous or paradoxical, it declines to decide and
marks the atom undefined — always yielding exactly one model, computable in polynomial time.

They agree whenever the program is well-behaved (stratified). To confirm, here is the
tweety/opus default program from §2 encoded propositionally — WFS returns a *total* model (no
undefined), matching what Clingo computed:

In [12]:
# penguin(opus) is a fact; flies(x) :- not abnormal(x); abnormal(opus) from penguin(opus)
stratified = [
    WRule("abnormal_opus", frozenset()),                  # opus is a penguin => abnormal
    WRule("flies_tweety",  frozenset({"abnormal_tweety"})),
    WRule("flies_opus",    frozenset({"abnormal_opus"})),
]
show_wfs("stratified default", stratified)
print()
print("No 'undefined' anywhere: on stratified programs WFS, ASP and Datalog all agree.")

stratified default: TRUE=['abnormal_opus', 'flies_tweety']  FALSE=['abnormal_tweety', 'flies_opus']  UNDEFINED=[]

No 'undefined' anywhere: on stratified programs WFS, ASP and Datalog all agree.


## Putting it together: the `typedlogic` profiles

`typedlogic` reifies these semantic choices as **profile** classes in `typedlogic.profiles`, so a
theory or solver can declare which assumptions it makes. The hierarchy encodes the relationships
we just demonstrated — e.g. `WellFoundedSemantics` and `ClassicPrologNegationAsFailure` are both
kinds of `ClosedWorld`, and `AnswerSetProgramming` is a `WellFoundedSemantics` that also permits
`MultipleModelSemantics` (the several answer sets we saw).

In [13]:
from typedlogic.profiles import (
    OpenWorld, ClosedWorld, SingleModelSemantics, MultipleModelSemantics,
    ClassicDatalog, ClassicPrologNegationAsFailure, WellFoundedSemantics,
    AnswerSetProgramming, Unrestricted,
)

profiles = [
    ("Classical FOL (Z3)",       Unrestricted()),
    ("Datalog",                  ClassicDatalog()),
    ("Prolog NAF",               ClassicPrologNegationAsFailure()),
    ("Well-Founded Semantics",   WellFoundedSemantics()),
    ("Answer Set Programming",   AnswerSetProgramming()),
]

hdr = f"{'profile':26} {'closed?':8} {'open?':6} {'single?':8} {'multi?':7}"
print(hdr); print("-" * len(hdr))
for label, p in profiles:
    print(f"{label:26} {str(p.impl(ClosedWorld)):8} {str(p.impl(OpenWorld)):6} "
          f"{str(p.impl(SingleModelSemantics)):8} {str(p.impl(MultipleModelSemantics)):7}")

profile                    closed?  open?  single?  multi? 
-----------------------------------------------------------
Classical FOL (Z3)         False    False  False    False  
Datalog                    True     False  True     False  
Prolog NAF                 True     False  False    False  
Well-Founded Semantics     True     False  False    False  
Answer Set Programming     True     False  False    True   


### Takeaways

- **Open vs closed world** decides what *silence* means: unknown, or false. Pick by whether your
  data is complete.
- **Negation as failure** gives cheap, powerful default reasoning — and with it
  **non-monotonicity**: new facts can retract old conclusions. Classical negation stays monotonic
  but can't express defaults.
- When negation loops, **ASP** enumerates stable worlds (two, or none), while **WFS** computes a
  single three-valued model that marks the ambiguity **undefined**. They coincide on stratified
  programs.
- `typedlogic.profiles` lets you *name* the assumption you rely on, so a mismatch between what you
  meant and what your solver does becomes explicit rather than a silent bug.

### Where to go next

- **Multiple models** — the [multiple-models notebook](multiple-models.ipynb) explores enumerating
  worlds with Clingo in more depth.
- **Open-world modelling** — the [OWL-DL tutorial](../tutorial/04-owldl.ipynb) is the open-world
  assumption applied to ontologies.
- **Solvers** — the [solver overview](../integrations/solvers/index.md) lists which backend
  implements which semantics.